In [ ]:
import math
import pickle
import numpy as np
F2 = GF(2)

In [ ]:
def extend_matrix_to_full_rank_optimized(C):
    F = C.base_ring()
    n_rows, n_cols = C.dimensions()
    if n_rows > n_cols:
        print(f"Error: The number of rows ({n_rows}) is greater than the number of columns ({n_cols}).")
        return None

    r = C.rank()
    print(f"Processing new matrix: shape {C.dimensions()}, rank {r}")

    if r < n_rows:
        print(f"Error: The rank of the input matrix ({r}) is less than its number of rows ({n_rows}).")
        print("This means the original row vectors are linearly dependent and cannot be extended to a full-rank square matrix containing all original rows.")
        raise ValueError("Input matrix does not have full row rank, operation cannot be completed.")

    if n_rows == n_cols:
         print("Matrix is already square and full rank.")
         return C

    num_needed = n_cols - n_rows
    print(f"Extending, need to add {num_needed} standard basis vectors...")

    identity_mat = identity_matrix(F, n_cols)
    big_matrix = C.stack(identity_mat)
    try:
        pivot_row_indices = big_matrix.pivot_rows()
    except Exception as e:
        print(f"Error when computing pivot_rows: {e}")
        print("Trying .rows() and .pivot_rows() ...")
        temp_mat_for_pivots = matrix(F, big_matrix.rows())
        pivot_row_indices = temp_mat_for_pivots.pivot_rows()
    rows_to_add_vectors = []
    for index in pivot_row_indices:
        if index >= n_rows:
            e_index = index - n_rows
            rows_to_add_vectors.append(identity_mat.row(e_index))
    if len(rows_to_add_vectors) != num_needed:
         print(f"Warning: Number of found rows ({len(rows_to_add_vectors)}) does not match expected ({num_needed}).")
         rows_to_add_vectors = rows_to_add_vectors[:num_needed]

    if not rows_to_add_vectors and num_needed > 0:
         print("Error: No rows found to add, although the matrix is not square.")
         return None
    M_added = matrix(F, rows_to_add_vectors)

    final_matrix = C.stack(M_added)
    print(f"Extension complete. Final shape: {final_matrix.dimensions()}")
    return final_matrix

def extend_matrix_to_square(C):
    F = C.base_ring()
    n_rows, n_cols = C.dimensions()
    r = C.rank()

    print(f"--- Processing matrix: shape {n_rows}x{n_cols}, rank {r} ---")

    if n_rows == n_cols:
        if r == n_rows:
            print("Matrix is already full-rank square.")
            return C
        else:
            raise ValueError(f"Matrix is {n_rows}x{n_cols} square but not full rank (rank={r}), cannot extend.")

    elif n_rows < n_cols:
        print(f"Matrix M < N ({n_rows} < {n_cols}). Will add {n_cols - n_rows} rows.")
        try:
            return extend_matrix_to_full_rank_optimized(C)
        except ValueError as e:
            print(f"Error: Cannot extend 'wide' matrix because it does not have full row rank.")
            print(f"(Original matrix rank {r}, rows {n_rows})")
            raise e

    else:
        print(f"Matrix M > N ({n_rows} > {n_cols}). Will add {n_rows - n_cols} columns (via transpose).")
        C_T = C.transpose()
        try:
            print(f"Processing transposed matrix (shape {C_T.dimensions()})...")
            M_T = extend_matrix_to_full_rank_optimized(C_T)
            print("Transpose extension complete, transposing back...")
            final_matrix = M_T.transpose()
            print(f"Extension complete. Final shape: {final_matrix.dimensions()}")
            return final_matrix
        except ValueError as e:
            print(f"Error: Cannot extend 'tall' matrix because it does not have full column rank.")
            print(f"(Original matrix rank {r}, columns {n_cols})")
            raise ValueError(f"Tall matrix does not have full column rank (rank={r}, n_cols={n_cols}), cannot extend.") from e

In [ ]:
A_matrix = Matrix(F2, np.load("   ", allow_pickle=True))
A_full = extend_matrix_to_square(A_matrix)
if A_full is not None:
    print(f"New matrix rows: {A_full.nrows()}, rank: {A_full.rank()}")

if A_full is None:
    raise ValueError("A_full is None, cannot save.")

n_rows, n_cols = A_full.dimensions()
fname = "163_C.txt"
with open(fname, "w") as f:
    for i in range(n_rows):
        row_str = ''.join(str(int(A_full[i, j])) for j in range(n_cols))
        f.write(row_str + "\n")

print(f"Saved as {fname}, size {n_rows}x{n_cols}, no spaces in each row.")